# Visión desde el Cielo: Reconstrucción 3D con Fotos Satelitales

**Semana 13 — Taller 4**

La idea de este taller es reconstruir el relieve de un terreno a partir de dos vistas del
mismo lugar tomadas desde posiciones ligeramente distintas, como harían dos cámaras de un
satélite o un par de fotos aéreas con solape. Es el mismo principio que usa nuestra visión:
cada ojo ve la escena desde un punto diferente y el cerebro deduce la profundidad a partir de
esa diferencia.

En visión por computador esa diferencia se llama **disparidad**: cuánto se desplaza un mismo
punto entre la imagen izquierda y la derecha. Los puntos más cercanos (más altos, en un
terreno visto desde arriba) se desplazan más; los lejanos, menos. A partir de la disparidad
se construye un mapa de elevación (un DEM, *Digital Elevation Model*) y con él una malla 3D.

### Sobre las imágenes de entrada

El enunciado parte de un par de imágenes `left_image.jpg` / `right_image.jpg`. Para que el
notebook sea autocontenido y reproducible —y para poder comparar el resultado contra un
relieve conocido— **generamos un par estéreo sintético**: partimos de un terreno con elevación
definida por nosotros, y creamos la vista derecha desplazando cada píxel según esa elevación.
Así tenemos un *ground truth* contra el cual medir qué tan bien funciona la reconstrucción.
El mismo código funciona con imágenes reales reemplazando el bloque de generación por
`cv2.imread(...)`.

### Contenido
1. Generación del par estéreo sintético
2. Cálculo de la disparidad con StereoSGBM
3. Mapa de disparidad y profundidad
4. Mapa de elevación (DEM) y malla 3D
5. Validación contra el relieve real
6. Conclusiones

## 0. Librerías

Usamos OpenCV para la correspondencia estéreo, NumPy para el manejo de matrices, Matplotlib
para los mapas 2D y Plotly para la superficie 3D interactiva. En Colab, `opencv-python`,
`numpy` y `matplotlib` ya vienen instalados; `plotly` también suele estar disponible, pero
dejamos la instalación por si acaso.

In [ ]:
# En Colab, descomentar si hiciera falta:
# !pip install opencv-python plotly --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

print("OpenCV:", cv2.__version__)

rng = np.random.default_rng(7)   # semilla fija para reproducibilidad
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Generación del par estéreo sintético

Construimos primero el relieve real del terreno como una suma de gaussianas (tres "colinas"
de distinto tamaño y altura). Esta matriz `elevacion_real` es lo que queremos recuperar al
final, pero el algoritmo estéreo nunca la verá directamente: solo verá las dos imágenes.

In [ ]:
H, W = 300, 400
yy, xx = np.mgrid[0:H, 0:W]

def colina(cx, cy, sx, sy, amp):
    """Una elevación gaussiana centrada en (cx, cy)."""
    return amp * np.exp(-(((xx - cx)**2) / (2 * sx**2) + ((yy - cy)**2) / (2 * sy**2)))

elevacion_real = (colina(120, 100, 60, 50, 1.0)
                  + colina(280, 200, 80, 60, 0.8)
                  + colina(200, 150, 120, 100, 0.4))
elevacion_real /= elevacion_real.max()   # normalizada a [0, 1]

plt.imshow(elevacion_real, cmap="terrain")
plt.colorbar(label="Elevación (normalizada)")
plt.title("Relieve real del terreno (lo que queremos reconstruir)")
plt.show()

Ahora generamos una textura de superficie (ruido suavizado mezclado con el propio
relieve, para que haya detalle visual que el algoritmo pueda emparejar) y construimos las dos
vistas. La imagen izquierda es la textura tal cual; la derecha se obtiene desplazando cada
píxel horizontalmente una cantidad proporcional a la elevación —ahí está codificada la
disparidad—. El desplazamiento se hace de forma vectorizada con `cv2.remap`.

In [ ]:
# Textura de la superficie: ruido suave + componente del relieve
base = cv2.GaussianBlur(rng.uniform(0.2, 0.8, (H, W)), (0, 0), sigmaX=3)
textura = np.clip(0.6 * base + 0.4 * elevacion_real, 0, 1)
imagen = (textura * 255).astype(np.uint8)

# La disparidad real es proporcional a la elevación
DISPARIDAD_MAX = 32
disparidad_real = (elevacion_real * DISPARIDAD_MAX).astype(np.float32)

imgL = imagen.copy()
# imgR(x) debe contener lo que en imgL está en (x + d): muestreamos con remap
map_x = (xx + disparidad_real).astype(np.float32)
map_y = yy.astype(np.float32)
imgR = cv2.remap(imgL, map_x, map_y,
                 interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(imgL, cmap="gray"); ax1.set_title("Imagen izquierda"); ax1.axis("off")
ax2.imshow(imgR, cmap="gray"); ax2.set_title("Imagen derecha"); ax2.axis("off")
plt.tight_layout()
plt.show()

## 2. Cálculo de la disparidad con StereoSGBM

Para la correspondencia estéreo usamos `StereoSGBM` (Semi-Global Block Matching) en lugar de
`StereoBM`. SGBM da mapas de disparidad más limpios y con menos huecos porque impone
suavidad considerando varias direcciones, lo que viene bien en terrenos donde la elevación
cambia de forma gradual.

Parámetros importantes:
- `numDisparities`: rango de disparidades a buscar (múltiplo de 16). Debe cubrir la disparidad
  máxima esperada.
- `blockSize`: tamaño de la ventana de comparación. Impar; valores pequeños captan más detalle
  pero son más sensibles al ruido.
- `P1`, `P2`: penalizaciones por cambios de disparidad; controlan la suavidad del resultado.

In [ ]:
block = 7
num_disp = 64   # múltiplo de 16, cubre la disparidad máxima (32) con margen

stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=num_disp,
    blockSize=block,
    P1=8 * block * block,
    P2=32 * block * block,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
)

# compute() devuelve la disparidad en punto fijo escalada por 16
disparidad = stereo.compute(imgL, imgR).astype(np.float32) / 16.0

valido = disparidad > 0
print(f"Píxeles con disparidad válida: {100 * valido.mean():.1f}%")
print(f"Rango de disparidad estimada: [{disparidad[valido].min():.1f}, {disparidad.max():.1f}]")

## 3. Mapa de disparidad y de profundidad

Visualizamos la disparidad estimada. Las zonas más claras corresponden a mayor disparidad,
es decir, a las partes más altas del terreno.

In [ ]:
plt.imshow(disparidad, cmap="inferno")
plt.colorbar(label="Disparidad (píxeles)")
plt.title("Mapa de disparidad estimado")
plt.show()

La profundidad es inversamente proporcional a la disparidad: a mayor disparidad, menor
distancia a la cámara. Aplicamos esa relación y recortamos los valores extremos que aparecen
donde la disparidad es muy baja (bordes y zonas sin emparejar).

In [ ]:
depth_map = 1.0 / (disparidad + 1e-6)   # profundidad inversa
depth_map[disparidad <= 0] = np.nan      # zonas sin dato válido
# recorte de valores extremos para visualización
limite = np.nanpercentile(depth_map, 95)
depth_map = np.clip(depth_map, None, limite)

plt.imshow(depth_map, cmap="viridis")
plt.colorbar(label="Profundidad relativa")
plt.title("Mapa de profundidad")
plt.show()

## 4. Mapa de elevación (DEM) y malla 3D

Para un terreno visto desde arriba, la elevación es lo opuesto a la profundidad: lo más alto
está más cerca de la cámara. Por eso el DEM se obtiene directamente de la disparidad (mayor
disparidad → mayor elevación). Suavizamos un poco para reducir el ruido y rellenamos los
huecos antes de construir la malla.

In [ ]:
# La elevación es proporcional a la disparidad
dem = disparidad.copy()
dem[disparidad <= 0] = np.nan

# Relleno de huecos por interpolación con la mediana local + suavizado
mascara = np.isnan(dem)
dem_relleno = dem.copy()
dem_relleno[mascara] = np.nanmedian(dem)
dem_suave = cv2.GaussianBlur(dem_relleno, (0, 0), sigmaX=2)

plt.imshow(dem_suave, cmap="terrain")
plt.colorbar(label="Elevación reconstruida")
plt.title("DEM reconstruido (mapa de elevación)")
plt.show()

Con el DEM ya podemos levantar la malla 3D. Usamos Plotly para una superficie
interactiva: en Colab se puede rotar, hacer zoom y mirar el relieve desde cualquier ángulo.
Submuestreamos un poco la malla para que la visualización sea fluida.

In [ ]:
paso = 2   # submuestreo para aligerar la malla
z = dem_suave[::paso, ::paso]
x = np.arange(0, W, paso)
y = np.arange(0, H, paso)

fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale="earth")])
fig.update_layout(
    title="Terreno reconstruido en 3D",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Elevación",
        aspectratio=dict(x=1, y=H / W, z=0.4),
    ),
    autosize=True, height=600,
)
fig.show()

### (Opcional) Malla 3D con la textura de la imagen original

Podemos colorear la superficie con la intensidad de la imagen izquierda en lugar de la altura,
de modo que la malla muestre el aspecto real del terreno sobre el relieve reconstruido. Esto
se logra pasando la textura en `surfacecolor`.

In [ ]:
textura_sub = imgL[::paso, ::paso]

fig_tex = go.Figure(data=[go.Surface(
    z=z, x=x, y=y,
    surfacecolor=textura_sub,   # color = textura, no altura
    colorscale="gray",
    showscale=False,
)])
fig_tex.update_layout(
    title="Terreno 3D con textura de la imagen original",
    scene=dict(aspectratio=dict(x=1, y=H / W, z=0.4)),
    autosize=True, height=600,
)
fig_tex.show()

## 5. Validación contra el relieve real

Como generamos el terreno nosotros, podemos medir objetivamente qué tan buena fue la
reconstrucción. Comparamos la disparidad estimada contra la real (que es proporcional a la
elevación verdadera) en los píxeles con dato válido, usando el coeficiente de correlación.

In [ ]:
mask = valido
corr = np.corrcoef(disparidad_real[mask], disparidad[mask])[0, 1]
print(f"Correlación entre disparidad real y estimada: {corr:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
im1 = ax1.imshow(disparidad_real, cmap="inferno")
ax1.set_title("Disparidad real (ground truth)"); plt.colorbar(im1, ax=ax1, fraction=0.046)
im2 = ax2.imshow(disparidad, cmap="inferno")
ax2.set_title("Disparidad estimada (SGBM)"); plt.colorbar(im2, ax=ax2, fraction=0.046)
plt.tight_layout()
plt.show()

Una correlación cercana a 1 indica que el relieve reconstruido reproduce fielmente la
forma del terreno real. Las diferencias se concentran en los bordes de la imagen y en las
zonas donde el solape entre las dos vistas es menor, que es justo donde el algoritmo tiene
menos información para emparejar.

## 6. Conclusiones

- A partir de dos vistas desplazadas de un mismo terreno se reconstruyó su relieve calculando
  la disparidad con `StereoSGBM`, sin medir nunca la altura directamente. La elevación es la
  variable que se infiere del desplazamiento entre imágenes.
- La validación contra el terreno real dio una correlación alta, lo que confirma que el
  pipeline (disparidad → profundidad/elevación → malla 3D) recupera correctamente la forma.
- SGBM resultó preferible a BM por entregar mapas más limpios y con menos huecos, a costa de
  algo más de cómputo. La calidad depende mucho de `numDisparities` y `blockSize`.
- Los puntos débiles aparecen en bordes y zonas de poco solape, donde falta información para
  emparejar. En imágenes satelitales reales se suman otros retos: rectificación de las vistas,
  iluminación distinta entre tomas y zonas sin textura (agua, nieve) difíciles de emparejar.
- El mismo flujo se aplica a fotogrametría real para generar modelos de elevación de terreno,
  reconstrucción urbana 3D y cartografía a partir de imágenes aéreas o de satélite.